# 05 — Trendonify API crude oil stock history

Trendonify embeds the full API crude stock-change chart history in an
`astro-island` HTML attribute called `props`.

The loader:

- performs one public HTTP GET;
- saves the raw HTML for reproducibility;
- finds the Astro island containing `chartData`;
- decodes Astro's serialized values;
- extracts `Date` and `Value`;
- interprets `Date` as the **week-ending/reference Friday**, not as the
  market release timestamp;
- keeps the full raw history and separately selects 2015–2016 to fill the
  gap before Myfxbook coverage.

Do not infer API release timestamps by adding four days. Holiday weeks can
shift the release schedule.


In [1]:
from pathlib import Path

import pandas as pd

import sys

sys.path.append('../../')

from src.data.trendonify_api_crude_loader import (
    fetch_trendonify_api_crude,
    select_model_period,
    select_trendonify_pre_myfxbook,
)


def find_project_root(
    start: Path | None = None,
) -> Path:
    current = (start or Path.cwd()).resolve()

    for candidate in [current, *current.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate

    msg = "Could not locate project root."
    raise FileNotFoundError(msg)


ROOT = find_project_root()

RAW_DIR = (
    ROOT
    / "data"
    / "external"
    / "trendonify"
)
PROCESSED_DIR = (
    ROOT
    / "data"
    / "processed"
    / "consensus"
)

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

RAW_HTML_PATH = (
    RAW_DIR
    / "api_crude_oil_stock_change.html"
)


## 1. Fetch and parse full history


In [2]:
result = fetch_trendonify_api_crude(
    raw_html_path=RAW_HTML_PATH,
    force=False,
)

trendonify_full = result.frame

print(
    "rows:",
    len(trendonify_full),
)
print(
    "range:",
    trendonify_full["reference_week_end"].min(),
    "->",
    trendonify_full["reference_week_end"].max(),
)
print(
    "fetched:",
    result.fetched_at_utc,
)

display(trendonify_full.head())
display(trendonify_full.tail())


rows: 753
range: 2012-03-23 00:00:00 -> 2026-10-30 00:00:00
fetched: 2026-11-06 23:07:17.977884+00:00


,reference_week_end,api_actual_million,country,original_reference_week_end,reference_date_adjusted,reference_date_adjustment_days,source,series,unit,fetched_at_utc
0,2012-03-23,3.60,united-states,2012-03-23,False,0,trendonify,api_crude_oil_stock_change,million_barrels,2026-11-06 23:07:17.977884+00:00
1,2012-03-30,7.85,united-states,2012-03-30,False,0,trendonify,api_crude_oil_stock_change,million_barrels,2026-11-06 23:07:17.977884+00:00
2,2012-04-06,6.58,united-states,2012-04-06,False,0,trendonify,api_crude_oil_stock_change,million_barrels,2026-11-06 23:07:17.977884+00:00
3,2012-04-13,3.41,united-states,2012-04-13,False,0,trendonify,api_crude_oil_stock_change,million_barrels,2026-11-06 23:07:17.977884+00:00
4,2012-04-20,-0.99,united-states,2012-04-20,False,0,trendonify,api_crude_oil_stock_change,million_barrels,2026-11-06 23:07:17.977884+00:00


,reference_week_end,api_actual_million,country,original_reference_week_end,reference_date_adjusted,reference_date_adjustment_days,source,series,unit,fetched_at_utc
748,2026-10-02,-2.090,united-states,2026-10-02,False,0,trendonify,api_crude_oil_stock_change,million_barrels,2026-11-06 23:07:17.977884+00:00
749,2026-10-09,-11.120,united-states,2026-10-09,False,0,trendonify,api_crude_oil_stock_change,million_barrels,2026-11-06 23:07:17.977884+00:00
750,2026-10-16,-4.930,united-states,2026-10-16,False,0,trendonify,api_crude_oil_stock_change,million_barrels,2026-11-06 23:07:17.977884+00:00
751,2026-10-23,-2.031,united-states,2026-10-23,False,0,trendonify,api_crude_oil_stock_change,million_barrels,2026-11-06 23:07:17.977884+00:00
752,2026-10-30,-5.267,united-states,2026-10-30,False,0,trendonify,api_crude_oil_stock_change,million_barrels,2026-11-06 23:07:17.977884+00:00


## 2. QA


In [3]:
assert not trendonify_full[
    "reference_week_end"
].duplicated().any()

assert trendonify_full[
    "api_actual_million"
].notna().all()

assert set(
    trendonify_full[
        "reference_week_end"
    ].dt.dayofweek.unique()
) == {4}

print(
    "All reference dates are Fridays."
)
print(
    "Full-history QA passed."
)


All reference dates are Fridays.
Full-history QA passed.


/var/folders/yf/82c7fyfs1m3c7t3l4p06rzp00000gn/T/ipykernel_50059/1900770409.py:12: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  ].dt.dayofweek.unique()


## 3. Inspect known overlap anomaly



In [4]:
anomaly = trendonify_full.loc[
    trendonify_full[
        "reference_week_end"
    ].eq(pd.Timestamp("2026-08-14"))
]

display(anomaly)


,reference_week_end,api_actual_million,country,original_reference_week_end,reference_date_adjusted,reference_date_adjustment_days,source,series,unit,fetched_at_utc
741,2026-08-14,-0.328,united-states,2026-08-14,False,0,trendonify,api_crude_oil_stock_change,million_barrels,2026-11-06 23:07:17.977884+00:00


In [7]:
trendonify_full.loc[trendonify_full['api_actual_million'] == -0.328, 'api_actual_million'] = -3.28

In [8]:
anomaly = trendonify_full.loc[
    trendonify_full[
        "reference_week_end"
    ].eq(pd.Timestamp("2026-08-14"))
]

display(anomaly)


,reference_week_end,api_actual_million,country,original_reference_week_end,reference_date_adjusted,reference_date_adjustment_days,source,series,unit,fetched_at_utc
741,2026-08-14,-3.28,united-states,2026-08-14,False,0,trendonify,api_crude_oil_stock_change,million_barrels,2026-11-06 23:07:17.977884+00:00


## 4. Select project periods


In [9]:
trendonify_model = select_model_period(
    trendonify_full,
    start_date="2015-01-01",
)

trendonify_2015_2016 = (
    select_trendonify_pre_myfxbook(
        trendonify_full
    )
)

print(
    "2015+ rows:",
    len(trendonify_model),
)
print(
    "2015-2016 rows:",
    len(trendonify_2015_2016),
)


2015+ rows: 618
2015-2016 rows: 105


## 5. Save source layer


In [10]:
full_path = (
    PROCESSED_DIR
    / "trendonify_api_crude_full_history.parquet"
)
gap_path = (
    PROCESSED_DIR
    / "trendonify_api_crude_2015_2016.parquet"
)

trendonify_full.to_parquet(
    full_path,
    index=False,
)
trendonify_2015_2016.to_parquet(
    gap_path,
    index=False,
)

print("saved:", full_path)
print("saved:", gap_path)


saved: /Users/ivanandreev/Desktop/ИИ/Нефть/oil-news-graph/data/processed/consensus/trendonify_api_crude_full_history.parquet
saved: /Users/ivanandreev/Desktop/ИИ/Нефть/oil-news-graph/data/processed/consensus/trendonify_api_crude_2015_2016.parquet
